# SP00 — SpatialData: The Modern Spatial Container

**Tools:** `spatialdata`, `spatialdata-io`, `spatialdata-plot`  
**Dataset:** Synthetic blobs (built-in) + mouse brain Visium (via spatialdata-io)  
**Paper:** [Marconato et al. 2024, Nature Methods](https://doi.org/10.1038/s41592-024-02212-x)

---

## The problem with AnnData+coords for spatial data

The approach in `theis_ecosystem/T02` — storing spatial coordinates in `adata.obsm['spatial']` and the H&E image in `adata.uns['spatial']` — works for simple Visium datasets but breaks down for:
- **Multi-scale images** (Visium HD tiles, high-res H&E at multiple zoom levels)
- **Multiple coordinate systems** (raw pixel coords vs. micron coords vs. registered multi-slide coords)
- **Transcript-level data** (Xenium, MERFISH — millions of individual transcript locations, not spots)
- **Cell segmentation** (polygon boundaries per cell, not just centroid coordinates)
- **Multi-sample / multi-slide** datasets (can't store 10 slides in one AnnData cleanly)

## What SpatialData provides

`SpatialData` is a FAIR, Zarr-backed container that unifies all spatial omics platforms:

```
SpatialData (on-disk: .zarr store)
  ├── images/        — multi-scale raster images (H&E, DAPI, IF channels)
  │     └── hires, lowres, fullres (OME-NGFF format)
  ├── labels/        — segmentation masks (cell labels, tissue regions)
  ├── shapes/        — vector geometries (cell polygons, tissue ROIs)
  │     └── GeoDataFrame with geometry column
  ├── points/        — transcript coordinates (Xenium, MERFISH, seqFISH)
  │     └── Dask DataFrame (lazy, handles 100M+ transcripts)
  └── tables/        — AnnData objects (expression per cell/spot)
        └── region_key links table rows to spatial elements
```

Everything is registered to a **coordinate system** (physical microns, pixel, global). Transformations between systems are stored as metadata.

**Key advantage:** It's backed by Zarr — you can have a 100GB dataset on disk and only load what you need.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import spatialdata as sd
from spatialdata import SpatialData
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=80, facecolor='white')
print(f'spatialdata {sd.__version__}')

## 1. Anatomy of a SpatialData Object

We start with the built-in `blobs()` synthetic dataset — it has all element types (images, labels, shapes, points, tables) so we can inspect each slot without downloading real data.

In [ ]:
# Built-in synthetic dataset with all element types
sdata = sd.datasets.blobs()
print(sdata)
print()
print('Images:  ', list(sdata.images.keys()))
print('Labels:  ', list(sdata.labels.keys()))
print('Shapes:  ', list(sdata.shapes.keys()))
print('Points:  ', list(sdata.points.keys()))
print('Tables:  ', list(sdata.tables.keys()))

In [ ]:
# Images: multi-scale xarray DataArray (OME-NGFF)
img = sdata.images['blobs_image']
print('Image type:', type(img))
print('Image shape:', img.shape)    # (C, Y, X) or (scale, C, Y, X)
print('Dims:', img.dims)
print('Coordinate systems:', sdata.coordinate_systems)

In [ ]:
# Labels: segmentation masks (integer array, each cell = unique int)
labels = sdata.labels['blobs_labels']
print('Labels shape:', labels.shape)
print('Unique cell IDs:', np.unique(labels.values)[:10], '...')

In [ ]:
# Shapes: GeoDataFrame with shapely polygon geometries
shapes = sdata.shapes['blobs_circles']
print('Shapes type:', type(shapes))
print(shapes.head())
print('CRS:', shapes.crs if hasattr(shapes, 'crs') else 'none')

In [ ]:
# Points: Dask DataFrame of individual transcript coordinates
# In real Xenium/MERFISH data this can be 10M+ rows
points = sdata.points['blobs_points']
print('Points type:', type(points))
print(points.head())
print('Columns:', points.columns.tolist())

In [ ]:
# Tables: AnnData objects linked to spatial elements
# region_key tells you which spatial element each row corresponds to
table = sdata.tables['table']
print('Table (AnnData):', table)
print('region_key:', table.uns.get('spatialdata_attrs', {}))
print('obs columns:', table.obs.columns.tolist())

## 2. Coordinate Systems and Transformations

Every spatial element is registered to a coordinate system. When you have multiple slides or multiple modalities, you can align them all to a `global` coordinate system.

In [ ]:
from spatialdata.transformations import get_transformation, set_transformation
from spatialdata.transformations import Identity, Scale, Translation, Sequence

# Get the transformation of an element to its coordinate system
transform = get_transformation(sdata.images['blobs_image'], to_coordinate_system='global')
print('Image transformation:', transform)

# Apply a scaling transformation (e.g. pixels → microns at 0.5 µm/pixel)
scale_transform = Scale(scale=[0.5, 0.5], axes=('x', 'y'))
set_transformation(sdata.images['blobs_image'], scale_transform, to_coordinate_system='microns')
print('Applied scale transform to microns coordinate system')

## 3. Visualization with spatialdata-plot

In [ ]:
# spatialdata-plot uses a chained .pl API, similar to scanpy's
try:
    import spatialdata_plot
    sdata.pl.render_images('blobs_image').pl.render_labels('blobs_labels').pl.show()
except ImportError:
    # Fallback: matplotlib directly
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    img_data = sdata.images['blobs_image'].values
    if img_data.ndim == 3:
        axes[0].imshow(img_data[0], cmap='gray')
    axes[0].set_title('Image channel 0')
    
    labels_data = sdata.labels['blobs_labels'].values
    axes[1].imshow(labels_data, cmap='tab20')
    axes[1].set_title('Cell segmentation labels')
    plt.tight_layout()
    plt.show()

## 4. Loading Real Visium Data with spatialdata-io

In practice, you load spatial data from platform-specific output folders. `spatialdata-io` provides loaders for all major platforms.

In [ ]:
# spatialdata-io loaders for all major platforms:
from spatialdata_io import (
    visium,      # 10x Visium (Cell Ranger output)
    xenium,      # 10x Xenium in situ
    merfish,     # Vizgen MERSCOPE / MERFISH
    slide_seq,   # Broad Slide-seq v2
    codex,       # Akoya CODEX (protein imaging)
)

# Example: load a Cell Ranger Visium output directory
# sdata = visium(
#     path='path/to/spaceranger/output/',
#     dataset_id='mouse_brain_s1',
# )
# print(sdata)

print('Platform loaders available in spatialdata-io:')
for name in ['visium', 'xenium', 'merfish', 'slide_seq', 'codex']:
    print(f'  spatialdata_io.{name}(path, dataset_id, ...)')

In [ ]:
# Convert existing AnnData + squidpy-style spatial coords to SpatialData
# (migration path from the old AnnData+obsm['spatial'] pattern)
import squidpy as sq
from spatialdata.models import TableModel, ShapesModel

adata = sq.datasets.visium_hne_adata()
print('Original AnnData:', adata)

# Extract spatial coordinates
coords = adata.obsm['spatial']
spot_diameter_fullres = adata.uns['spatial'][list(adata.uns['spatial'].keys())[0]]['scalefactors']['spot_diameter_fullres']

# Wrap spot centers as circles in SpatialData shapes
import geopandas as gpd
from shapely.geometry import Point
spot_shapes = gpd.GeoDataFrame(
    {'geometry': [Point(x, y) for x, y in coords],
     'radius': spot_diameter_fullres / 2},
    index=adata.obs_names,
)
spot_shapes_sd = ShapesModel.parse(spot_shapes)

# Wrap expression table
table_sd = TableModel.parse(
    adata,
    region='visium_spots',
    region_key='_region',
    instance_key='_instance_id',
)
adata.obs['_region'] = 'visium_spots'
adata.obs['_instance_id'] = np.arange(adata.n_obs)
table_sd = TableModel.parse(adata, region='visium_spots',
                             region_key='_region', instance_key='_instance_id')

sdata_visium = SpatialData(
    shapes={'visium_spots': spot_shapes_sd},
    tables={'table': table_sd},
)
print('\nConverted to SpatialData:', sdata_visium)

In [ ]:
# Save SpatialData to disk (Zarr format)
out_path = '../data/visium_sdata.zarr'
sdata_visium.write(out_path, overwrite=True)
print(f'Saved to {out_path}')

# Reload (lazy — doesn't load arrays into RAM)
sdata_reloaded = sd.read_zarr(out_path)
print('Reloaded:', sdata_reloaded)
print('Images loaded?', bool(sdata_reloaded.images))  # lazy if backed

## 5. SpatialData ↔ Squidpy Interoperability

Squidpy functions accept both SpatialData and plain AnnData. The `sdata.tables['table']` AnnData is the interface point.

In [ ]:
# Access the AnnData from SpatialData for use with scanpy/squidpy
adata_from_sdata = sdata_visium.tables['table']
print('AnnData from SpatialData:', adata_from_sdata)

# All scanpy/squidpy functions work on this
sc.pp.filter_genes(adata_from_sdata, min_cells=3)
sc.pp.normalize_total(adata_from_sdata, target_sum=1e4)
sc.pp.log1p(adata_from_sdata)
sc.pp.pca(adata_from_sdata)

# Spatial coordinates are in .obsm['spatial'] (same as before)
print('Spatial coords:', adata_from_sdata.obsm.get('spatial', 'not found'))

---
## Summary

| Element | Type | Access | Stores |
|---------|------|--------|--------|
| `sdata.images` | Multi-scale xarray | `sdata.images['name']` | H&E, DAPI, IF channels |
| `sdata.labels` | xarray | `sdata.labels['name']` | Cell/tissue segmentation masks |
| `sdata.shapes` | GeoDataFrame | `sdata.shapes['name']` | Cell polygons, tissue ROIs |
| `sdata.points` | Dask DataFrame | `sdata.points['name']` | Transcript coordinates (Xenium/MERFISH) |
| `sdata.tables` | AnnData dict | `sdata.tables['table']` | Gene expression per cell/spot |

**Key operations:**
```python
sdata = sd.read_zarr('path.zarr')           # lazy load
sdata = spatialdata_io.visium('spaceranger_out/')  # from Cell Ranger
sdata.write('output.zarr')                   # save to disk
adata = sdata.tables['table']               # get AnnData for scanpy/squidpy
```

**Next:** SP01 — Visium analysis in depth (spatial QC, regional DE, image features).